# SemEval-2020 Task 7: Assessing Humor in Edited News Headlines | Subtask 1: Estimate the funniness of an edited headline on a 0-3 humor scale

**Team 1:** Matthew Vu, Vu Doan, Prapurna Penmathsa, and Amrisha Dwivedi

**Description:** The code in this Jupyter Notebook file performs all the necessary operations to accomplish SemEval-2020 Task 7: Assessing Humor in Edited News Headlines | Subtask 1: Estimate the funniness of an edited headline on a 0-3 humor scale.

**Acknowledgements:** This code is heavily based on the work on the work of Nabil Hossain (https://github.com/n-hossain/semeval-2020-task-7-humicroedit/tree/master) and Salih Tuncer (https://github.com/SalihTuncer/AssessHumor).

## Step 0: Import libraries

In [38]:
import pandas as pd
import numpy as np
import sys
import os
import tensorflow as tf
from transformers import AlbertTokenizer

TypeError: Descriptors cannot be created directly.
If this call came from a _pb2.py file, your generated code is out of date and must be regenerated with protoc >= 3.19.0.
If you cannot immediately regenerate your protos, some other possible workarounds are:
 1. Downgrade the protobuf package to 3.20.x or lower.
 2. Set PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python (but this will use pure-Python parsing and will be much slower).

More information: https://developers.google.com/protocol-buffers/docs/news/2022-05-06#python-updates

## Step 1: Baseline RMSE values

Recall, we have a validation set called "dev.csv" that is used to develop models, tune hyperparameters, and choose the optimal model out of a series of possible models.  Also, we have a test set called "test.csv" that is used to evaluate the final, best model to get an estimate on how well the chosen model will perform on unseen data in the future.

So, the first step in our analysis is to compute two baseline root mean square error (RMSE) values.  One baseline RMSE value is equal to the validation set's ("dev.csv") RMSE when the predicted funniness score of all edited news headlines in that validation set ("dev.csv") is equal to the average mean funniness score of all edited news headlines in the training set ("train.csv").  

The other baseline RMSE value is equal to the test set's ("test.csv") RMSE when the predicted funniness score of all edited news headlines in the test set ("test.csv") is equal to the average mean funniness score of all edited news headlines in the training set ("train.csv").

We want our chosen model to perform better than both the validation and test set baseline RMSE values.

In [7]:
# this function reads in the training set and the test/validation set 
# and writes an output file that is a copy of the test/validation set
# with the baseline predictions
# inputs:
# train_loc -> directory of the training set
# test_loc -> directory of the test/validation set
# output_name -> name of the desired output file
def baseline_dataset(train_loc, test_loc, output_name):
    # read in the training and test sets
    train = pd.read_csv(train_loc)    
    test = pd.read_csv(test_loc)
    
    # make predictions = average of meanGrade
    pred = np.mean(train['meanGrade'])    
    test['pred'] = pred
    
    # create and write to output file
    output = test[['id','pred']] 
    out_loc = f'datasets/subtask1/baseline/{output_name}'
    output.to_csv(out_loc, index=False)
    
    # show that the output is written
    print(f'{output_name} successfully written.')

In [8]:
# this function reads in the test/validation set and a dataset containing the baseline 
# predictions to compute the baseline RMSE of that test/validation set
# and returns the baseline RMSE
def baseline_RMSE(truth_loc, prediction_loc):
    # read in the true values
    truth = pd.read_csv(truth_loc, usecols=['id','meanGrade'])
    # read in the baseline predictions
    pred = pd.read_csv(prediction_loc, usecols=['id','pred'])
    
    assert(sorted(truth.id) == sorted(pred.id)),"ID mismatch between ground truth and prediction!"
    
    # compute the baseline RMSE
    data = pd.merge(truth,pred)
    rmse = np.sqrt(np.mean((data['meanGrade'] - data['pred'])**2))
    
    # return baseline RMSE
    return rmse

In [9]:
# create validation set baseline dataset
baseline_dataset('datasets/subtask1/train.csv', 'datasets/subtask1/dev.csv', 'dev_baseline.csv')

dev_baseline.csv successfully written.


In [18]:
# compute the output the validation set RMSE value
validation_baseline = baseline_RMSE('datasets/subtask1/dev.csv', 'datasets/subtask1/baseline/dev_baseline.csv')
print(f'Baseline validation set RMSE = {validation_baseline:.3f}')

Baseline validation set RMSE = 0.578


In [11]:
# create test set baseline dataset
baseline_dataset('datasets/subtask1/train.csv', 'datasets/subtask1/test.csv', 'test_baseline.csv')

test_baseline.csv successfully written.


In [19]:
# compute the output the test set RMSE value
test_baseline = baseline_RMSE('datasets/subtask1/test.csv', 'datasets/subtask1/baseline/test_baseline.csv')
print(f'Baseline test set RMSE = {test_baseline:.3f}')

Baseline test set RMSE = 0.575


We want our final model to get a better baseline validation and test set RMSE of 0.578 and 0.575, respectively.

In [ ]:
## Step 2: 